# Regresión logística

In [ ]:
from datetime import datetime
from scipy.sparse import load_npz
from sklearn.metrics import (
    confusion_matrix,
    ConfusionMatrixDisplay,
    f1_score,
    precision_score,
    recall_score,
    RocCurveDisplay
)
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
import matplotlib.pyplot as plt
import pandas as pd

Se carga dataset.

In [ ]:
X_train = load_npz('datasets/X_train.npz')
X_test = load_npz('datasets/X_test.npz')

y_train = pd.read_csv('datasets/y_train.csv')['label']
y_test = pd.read_csv('datasets/y_test.csv')['label']

In [ ]:
scaler = StandardScaler(with_mean=False)

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [ ]:
logreg_model = LogisticRegression(max_iter=1000, solver='liblinear')

start_train = datetime.now()
logreg_model.fit(X_train_scaled, y_train)
train_duration = datetime.now() - start_train

start_pred = datetime.now()
y_pred = logreg_model.predict(X_test_scaled)
predict_duration = datetime.now() - start_pred

#### Métricas

Se evalúa.

In [ ]:
ConfusionMatrixDisplay.from_predictions(y_test, y_pred, display_labels=['Legítimo', 'Phishing'], cmap='Blues')

In [ ]:
cm_logreg = confusion_matrix(y_test, y_pred)
P = cm_logreg[1, :].sum()
N = cm_logreg[0, :].sum()
TP = cm_logreg[1, 1]
TN = cm_logreg[0, 0]

TPR_logreg = TP / P
TNR_logreg = TN / N
balanced_accuracy_logreg = (TPR_logreg + TNR_logreg) / 2

precision_logreg = precision_score(y_test, y_pred, zero_division=0)
recall_logreg = recall_score(y_test, y_pred, zero_division=0)
f1_logreg = f1_score(y_test, y_pred, zero_division=0)

print(f"Exactitud balanceada:    {balanced_accuracy_logreg:.4f}")
print(f"Precisión:               {precision_logreg:.4f}")
print(f"Sensibilidad (Recall):   {TPR_logreg:.4f}")
print(f"F1-score:                {f1_logreg:.4f}")
print(f"Especificidad:           {TNR_logreg:.4f}")
print(f"Recuperación (Recall):   {recall_logreg:.4f}")
print(f"Tiempo de entrenamiento: {train_duration}")
print(f"Tiempo de predicción:    {predict_duration}")

In [ ]:
y_scores = logreg_model.predict_proba(X_test_scaled)[:, 1]

RocCurveDisplay.from_predictions(y_test, y_scores)